# Exp 17 — Подготовка русских датасетов

Запускает `prepare.prepare_*` для каждого датасета и показывает результат.
Каждая функция парсит JSON-колонки в поля, делает multi-hot top-30 для массивов опций,
дропает `id` и константные колонки, сохраняет `data/raw_ru/<name>/clean.parquet`.

Если kagglehub просит логин — запусти `kagglehub.login()` в отдельной ячейке.

In [25]:
from __future__ import annotations

import sys
from pathlib import Path

# чтобы `import prepare` сработал из ноутбука в experiments/17/
HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 200)

import prepare
from importlib import reload
reload(prepare)

<module 'prepare' from '/home/user/Lysenko/Diplom/TableUnifier/experiments/17/prepare.py'>

In [26]:
def summarize(df: pd.DataFrame, name: str) -> None:
    print(f"=== {name} ===")
    print(f"shape: {df.shape}")
    print(f"memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

    def safe_nunique(s):
        try:
            return s.nunique(dropna=True)
        except TypeError:
            return -1

    stats = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_unique": df.apply(safe_nunique),
        "n_null": df.isna().sum(),
        "null_%": (df.isna().mean() * 100).round(1),
    })
    print(stats.to_string())


def show_samples(df: pd.DataFrame, name: str, n: int = 1, max_len: int = 160) -> None:
    print(f"=== samples: {name} ===")
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]) or pd.api.types.is_bool_dtype(df[col]):
            continue
        vals = df[col].dropna().head(n).tolist()
        if not vals:
            continue
        print(f"\n· {col}")
        for v in vals:
            s = str(v).replace("\n", " ")
            if len(s) > max_len:
                s = s[:max_len] + "…"
            print(f"    {s}")

## 1. Lamoda

In [27]:
df_lamoda = prepare.prepare_lamoda()
summarize(df_lamoda, "lamoda")

=== lamoda ===
shape: (10338, 50)
memory: 24.6 MB
                                      dtype  n_unique  n_null  null_%
Title                                object      1979       0     0.0
Brand                                object      1093       0     0.0
Price                                object      2707    3373    32.6
Image                                object     10337       0     0.0
about.Состав, %                      object      2407      36     0.3
about.Сезон                          object         9      53     0.5
about.Размер товара на модели        object       303    2077    20.1
about.Длина по внутреннему шву (см)  object        69    8742    84.6
about.Длина по боковому шву          object        88    8807    85.2
about.Ширина по низу                 object        40    9332    90.3
about.Высота                         object        23    9635    93.2
about.Параметры модели               object       587    5006    48.4
about.Артикул                        obj

In [28]:
show_samples(df_lamoda, "lamoda")

=== samples: lamoda ===

· Title
    Брюки спортивные

· Brand
    O'STIN

· Price
    680 ₽

· Image
    //a.lmcdn.ru/img600x866/M/P/MP002XM0BBEG_24488927_1_v1_2x.jpg

· about.Состав, %
    Хлопок - 60%, Полиэстер - 40%

· about.Сезон
    мульти

· about.Размер товара на модели
    M INT

· about.Длина по внутреннему шву (см)
    71 см

· about.Длина по боковому шву
    103 см

· about.Ширина по низу
    17 см

· about.Высота
    35 см

· about.Параметры модели
    100 - 84 -94

· about.Артикул
    MP002XM0BBEG

· about.Рост модели на фото
    189 см

· about.Материал подкладки, %
    без подкладки

· about.Цвет
    черный

· about.Узор
    однотонный

· about.Застежка
    пуговицы

· about.Страна производства
    Китай

· about.Гарантийный срок
    30 дней

· about.Длина (см)
    65 см

· about.Длина рукава (см)
    65 см

· about.Вырез/воротник
    поло

· about.Комплектация
    Футболка

· about.Экосостав
    данный товар имеет натуральный или органический состав

· about.Флисовая 

In [29]:
df_lamoda.head(2)

,Title,Brand,Price,Image,"about.Состав, %",about.Сезон,about.Размер товара на модели,about.Длина по внутреннему шву (см),about.Длина по боковому шву,about.Ширина по низу,about.Высота,about.Параметры модели,about.Артикул,about.Рост модели на фото,"about.Материал подкладки, %",about.Цвет,about.Узор,about.Застежка,about.Страна производства,about.Гарантийный срок,about.Длина (см),about.Длина рукава (см),about.Вырез/воротник,about.Комплектация,about.Экосостав,about.Флисовая подкладка,about.Утеплитель,about.Фасон,about.Вид спорта,about.Сезон 2,about.Тип трикотажа,about.Рисунок,about.Тип силуэта,about.Внешние карманы,about.Тип ткани,about.Детали,about.Бережно к животным,about.Внутренние карманы,about.Капюшон,about.Технологии,about.Мембрана,about.Посадка,about.Тип плавок,about.Фасон брюк,about.Косточки,about.Чашечки,about.Бретели,about.Push up,about.Количество Den,about.Тип юбки
0,Брюки спортивные,O'STIN,680 ₽,//a.lmcdn.ru/img600x866/M/P/MP002XM0BBEG_24488927_1_v1_2x.jpg,"Хлопок - 60%, Полиэстер - 40%",мульти,M INT,71 см,103 см,17 см,35 см,100 - 84 -94,MP002XM0BBEG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Брюки спортивные,O'STIN,680 ₽,//a.lmcdn.ru/img600x866/M/P/MP002XM0BBEL_24107566_1_v1_2x.jpg,"Хлопок - 60%, Полиэстер - 40%",мульти,M INT,71 см,100 см,NaN,31 см,103 - 78 - 98,MP002XM0BBEL,189 см,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. cars_ru (market)

In [30]:
df_cars_ru = prepare.prepare_cars_ru()
summarize(df_cars_ru, "cars_ru")

=== cars_ru ===
shape: (100000, 54)
memory: 339.3 MB
                            dtype  n_unique  n_null  null_%
price_rub                 float64      3257   19991    20.0
mark                       object       193       0     0.0
model                      object      1890       0     0.0
generation                 object      1082    9862     9.9
configuration              object       268       0     0.0
complectation              object      2642   68456    68.5
body_type                  object        26       0     0.0
color                      object        16       8     0.0
displacement              float64        73       0     0.0
drive_type                 object         3       0     0.0
engine_type                object         5       0     0.0
horse_power                 int64       470       0     0.0
transmission               object         4       0     0.0
wheel                      object         2       0     0.0
km_age                    float64     16098    

In [31]:
show_samples(df_cars_ru, "cars_ru")

=== samples: cars_ru ===

· mark
    Kia

· model
    Sportage

· generation
    IV Рестайлинг

· configuration
    Внедорожник 5 дв.

· complectation
    Prestige Black Edition

· body_type
    пятидверный внедорожник

· color
    чёрный

· drive_type
    полный

· engine_type
    бензин

· transmission
    автомат

· wheel
    левый

· condition
    среднее

· pts
    оригинал

· seller_type
    частник

· region
    Вологодская область

· city
    Череповец

· address
    Россия, Вологодская область, 

· description
    Автомобиль в отличном состоянии, вложений никаких не требует. 2,4 атмосферный двигатель на обычном автомате Полный привод! Куплен не в кредит. Своевременно обсл…


In [32]:
df_cars_ru.head(2)

,price_rub,mark,model,generation,configuration,complectation,body_type,color,displacement,drive_type,engine_type,horse_power,transmission,wheel,km_age,year,owners_count,condition,pts,seller_type,region,city,address,description,opt_airbag-driver,opt_electro-window-front,opt_lock,opt_immo,opt_abs,opt_wheel-configuration1,opt_airbag-passenger,opt_electro-mirrors,opt_mirrors-heat,opt_seat-transformation,opt_computer,opt_electro-window-back,opt_front-seats-heat,opt_isofix,opt_wheel-power,opt_wheel-configuration2,opt_airbag-side,opt_esp,opt_ptf,opt_fabric-seats,opt_audiosystem-cd,opt_wheel-leather,opt_multi-wheel,opt_usb,opt_bluetooth,opt_airbag-curtain,opt_cruise-control,opt_front-centre-armrest,opt_12v-socket,opt_light-sensor
0,2200000.0,Kia,Sportage,IV Рестайлинг,Внедорожник 5 дв.,Prestige Black Edition,пятидверный внедорожник,чёрный,2.4,полный,бензин,184,автомат,левый,51150.0,2020,1,среднее,оригинал,частник,Вологодская область,Череповец,"Россия, Вологодская область,","Автомобиль в отличном состоянии, вложений никаких не требует. 2,4 атмосферный двигатель на обычном автомате Полный привод! Куплен не в кредит. Своевременно обслуживался. Прошёл все ТО. Есть сервис...",True,True,True,True,True,True,True,False,False,True,True,True,False,True,False,True,True,True,True,False,True,True,True,False,True,True,True,True,True,True
1,300000.0,Ford,Maverick,III,Внедорожник 5 дв.,NaN,пятидверный внедорожник,серебристый,3.0,полный,бензин,197,автомат,левый,250000.0,2001,4,среднее,дубликат,частник,Краснодарский край,Сочи,Теневой переулок,"Есть жизненные повреждения, автомобиль рабочий, состояние соответствующее, по ходовой и технике все хорошо, езжу каждый день. руки естественно приложить есть куда",False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


## 3. Ozon favorites

In [33]:
df_ozon = prepare.prepare_ozon()
summarize(df_ozon, "ozon")

/home/user/Lysenko/Diplom/TableUnifier/experiments/17/prepare.py:185: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(parts, ignore_index=True)


=== ozon ===
shape: (333738, 14)
memory: 382.2 MB
                                                          dtype  n_unique  n_null  null_%
Название товара                                          object    170160     946     0.3
Бренд                                                    object     20991   27500     8.2
Ссылка на товар                                          object    198493       0     0.0
Категория 1 уровня                                       object        25       0     0.0
Категория 2 уровня                                       object       188       0     0.0
Категория 3 уровня                                       object       915       0     0.0
Категория 4 уровня                                       object      5003       0     0.0
Количество добавлений в избранное, 30 дней              float64      1792   23749     7.1
Количество добавлений в избранное, все время     datetime64[ns]       222  323738    97.0
Последнее появление в наличии                    d

In [34]:
show_samples(df_ozon, "ozon")

=== samples: ozon ===

· Название товара
    Дневник школьный, Кошечка, розовый

· Бренд
    bumbel

· Ссылка на товар
    https://www.ozon.ru/product/244927827

· Категория 1 уровня
    Товары для мам и детей

· Категория 2 уровня
    Товары для школы, канцелярия, оборудование

· Категория 3 уровня
    Бумага офисная и бумажная продукция

· Категория 4 уровня
    Школьный дневник

· Количество добавлений в избранное, все время
    2021-06-21 00:00:00

· Последнее появление в наличии
    2020-12-09 00:00:00

· __source_file
    2021-07-12_opendata_datasetfavorites_2021-06-12_2021-07-11.xlsx


In [35]:
df_ozon.head(2)

,Название товара,Бренд,Ссылка на товар,Категория 1 уровня,Категория 2 уровня,Категория 3 уровня,Категория 4 уровня,"Количество добавлений в избранное, 30 дней","Количество добавлений в избранное, все время",Последнее появление в наличии,__source_file,"Количество добавлений в избранное, декабрь 2020","Количество добавлений в избранное, ноябрь 2020","Количество добавлений в избранное, январь 2021"
0,"Дневник школьный, Кошечка, розовый",bumbel,https://www.ozon.ru/product/244927827,Товары для мам и детей,"Товары для школы, канцелярия, оборудование",Бумага офисная и бумажная продукция,Школьный дневник,3280.0,2021-06-21,NaT,2021-07-12_opendata_datasetfavorites_2021-06-12_2021-07-11.xlsx,NaN,NaN,NaN
1,"Игровая консоль PlayStation 5 Digital Edition, белый",PlayStation,https://www.ozon.ru/product/178715781,ТВ и аудио,Игровые приставки и аксессуары TV&Audio,Игровая приставка TV&Audio,Игровая приставка,3059.0,2021-06-24,NaT,2021-07-12_opendata_datasetfavorites_2021-06-12_2021-07-11.xlsx,NaN,NaN,NaN


## 4. auto.ru parsed

In [36]:
df_auto_ru = prepare.prepare_auto_ru()
summarize(df_auto_ru, "auto_ru")

/home/user/Lysenko/Diplom/TableUnifier/experiments/17/prepare.py:61: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)


=== auto_ru ===
shape: (70896, 86)
memory: 471.4 MB
                                        dtype  n_unique  n_null  null_%
bodyType                               object        25       0     0.0
brand                                  object        12       0     0.0
car_url                                object     60424       0     0.0
color                                  object        16       0     0.0
description                            object     52022    1000     1.4
engineDisplacement                    float64        56      55     0.1
enginePower                           float64       328       0     0.0
fuelType                               object         5       0     0.0
image                                  object     70536       0     0.0
mileage                                 int64     15830       0     0.0
modelDate                             float64        68       0     0.0
model_name                             object       733       0     0.0
name        

In [37]:
show_samples(df_auto_ru, "auto_ru")

=== samples: auto_ru ===

· bodyType
    лифтбек

· brand
    SKODA

· car_url
    https://auto.ru/cars/used/sale/skoda/octavia/1100575026-c780dc09/

· color
    синий

· description
    Все автомобили, представленные в продаже, проходят тщательную проверку по более 40 параметрам. Предоставляем гарантию юридической чистоты, а так же год техничес…

· fuelType
    бензин

· image
    https://autoru.naydex.net/o9DBXQ270/5ac010hAY0/Xkcrbmf2u0IghxJHqVi5dGL7OcugpPbM0sYLDhB9YWw7CxRKU17ysuJYxu9oaUHn7ahNSrqiKwm-CQDyDolDeEoEc3J49fgWYNYBUbQC7D96sj6K9…

· model_name
    OCTAVIA

· name
    1.2 AMT (105 л.с.)

· priceCurrency
    RUB

· sell_id
    1100575026

· vehicleConfiguration
    LIFTBACK ROBOT 1.2

· vehicleTransmission
    роботизированная

· vendor
    EUROPEAN

· Владельцы
    3 или более

· Владение
    3 года и 2 месяца

· ПТС
    Оригинал

· Привод
    передний

· Руль
    Левый

· model_info.code
    OCTAVIA

· model_info.name
    Octavia

· model_info.ru_name
    Октавия

· model_in

In [38]:
df_auto_ru.head(2)

,bodyType,brand,car_url,color,description,engineDisplacement,enginePower,fuelType,image,mileage,modelDate,model_name,name,numberOfDoors,parsing_unixtime,priceCurrency,productionDate,sell_id,vehicleConfiguration,vehicleTransmission,vendor,Владельцы,Владение,ПТС,Привод,Руль,is_train,price,model_info.code,model_info.name,model_info.ru_name,model_info.nameplate.code,model_info.nameplate.name,model_info.nameplate.semantic_url,super_gen.id,super_gen.displacement,super_gen.engine_type,super_gen.gear_type,super_gen.transmission,super_gen.power,super_gen.power_kvt,super_gen.human_name,super_gen.acceleration,super_gen.clearance_min,super_gen.fuel_rate,super_gen.nameplate,super_gen.clearance_max,super_gen.name,super_gen.ru_name,super_gen.year_from,super_gen.year_to,super_gen.price_segment,complectation_dict.id,complectation_dict.name,complectation_dict.available_options,complectation_dict.vendor_colors,equip_lock,equip_abs,equip_electro-mirrors,equip_airbag-driver,equip_computer,equip_electro-window-front,equip_airbag-passenger,equip_front-seats-heat,equip_immo,equip_electro-window-back,equip_ptf,equip_airbag-side,equip_esp,equip_mirrors-heat,equip_aux,equip_wheel-leather,equip_usb,equip_multi-wheel,equip_front-centre-armrest,equip_wheel-configuration1,equip_wheel-power,equip_rain-sensor,equip_light-sensor,equip_12v-socket,equip_bluetooth,equip_cruise-control,equip_isofix,equip_wheel-configuration2,equip_seat-transformation,equip_light-cleaner
0,лифтбек,SKODA,https://auto.ru/cars/used/sale/skoda/octavia/1100575026-c780dc09/,синий,"Все автомобили, представленные в продаже, проходят тщательную проверку по более 40 параметрам. Предоставляем гарантию юридической чистоты, а так же год технической гарантии на двигатель и КПП. Бес...",1.2,105.0,бензин,https://autoru.naydex.net/o9DBXQ270/5ac010hAY0/Xkcrbmf2u0IghxJHqVi5dGL7OcugpPbM0sYLDhB9YWw7CxRKU17ysuJYxu9oaUHn7ahNSrqiKwm-CQDyDolDeEoEc3J49fgWYNYBUbQC7D96sj6K9_O-mo6XT34oWVQDBTEybGZikaX4X4bwLyUuj...,74000,2013.0,OCTAVIA,1.2 AMT (105 л.с.),5.0,1603226273,RUB,2014,1100575026,LIFTBACK ROBOT 1.2,роботизированная,EUROPEAN,3 или более,NaN,Оригинал,передний,Левый,False,0,OCTAVIA,Octavia,Октавия,,,,10373605,1197.0,GASOLINE,FORWARD_CONTROL,ROBOT,105.0,77.0,1.2 AMT (105 л.с.),10.5,155.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True,True,True,True,True,True,False,True,True,True,False,False,True,False,False,False,False,True,True,True,False,True,False,False,True,True,True,False
1,лифтбек,SKODA,https://auto.ru/cars/used/sale/skoda/octavia/1100549428-595cadf2/,чёрный,ЛОТ: 01217195\nАвтопрага Север\nДанный автомобиль прошел диагностику по 147 пунктам и имеет сертификат технической гарантии.\n\n\nВы можете получить скидку на данный автомобиль до 50000 рублей. По...,1.6,110.0,бензин,https://autoru.naydex.net/o9DBXQ270/5ac010hAY0/Xkcrbmf2u0IghxJHqVi5dGL7OcugpPbM0sYLDhB9YWw7CxRKU17ysuJYxu9kY0fj5q5NROmsLgu7CQ34WY0UI0pVdyN6pahANodRBrYA5TJ-4z2K9_O-mo6XT34oWVQDBTEybGZikaX4X4bwLyUuj...,60563,2017.0,OCTAVIA,1.6 MT (110 л.с.),5.0,1603226277,RUB,2017,1100549428,LIFTBACK MECHANICAL 1.6,механическая,EUROPEAN,1 владелец,NaN,Оригинал,передний,Левый,False,0,OCTAVIA,Octavia,Октавия,,,,20913311,1598.0,GASOLINE,FORWARD_CONTROL,MECHANICAL,110.0,81.0,1.6 MT (110 л.с.),10.8,156.0,6.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True,True,True,False,True,True,True,True,True,True,True,True,False,False,True,False,False,False,True,True,True,False,True,True,True,True,False,False


## 5. auto.ru 14-11-2020

In [39]:
df_auto_ru_2020 = prepare.prepare_auto_ru_2020()
summarize(df_auto_ru_2020, "auto_ru_2020")

=== auto_ru_2020 ===
shape: (77449, 85)
memory: 561.4 MB
                                        dtype  n_unique  n_null  null_%
bodyType                               object       166       2     0.0
brand                                  object        36       0     0.0
color                                  object        16       0     0.0
description                            object     61684    2667     3.4
engineDisplacement                     object       103       0     0.0
enginePower                           float64       386       2     0.0
fuelType                               object         7       0     0.0
image                                   int64        47       0     0.0
mileage                                 int64     18645       0     0.0
modelDate                             float64        73       2     0.0
model_name                             object      1064       0     0.0
name                                   object      4333       2     0.0
numberO

In [40]:
show_samples(df_auto_ru_2020, "auto_ru_2020")

=== samples: auto_ru_2020 ===

· bodyType
    Седан

· brand
    AUDI

· color
    97948F

· description
    Машина на полном ходу Состояние хорошее С документами полный порядок(кристально чистая) Всего было 3 владельца  По всем вопросам ПИСАТЬ в личные сообщения (не з…

· engineDisplacement
    2.3

· fuelType
    бензин

· model_name
    100

· name
    2.3 MT (133 л.с.)

· start_date
    2020-11-13T23:31:09Z

· vehicleConfiguration
    SEDAN MECHANICAL 2.3

· vehicleTransmission
    MECHANICAL

· vendor
    EUROPEAN

· ПТС
    ORIGINAL

· Привод
    передний

· Руль
    LEFT

· model_info.code
    100

· model_info.name
    100

· model_info.ru_name
    100

· model_info.nameplate.code
    

· model_info.nameplate.name
    

· model_info.nameplate.semantic_url
    

· super_gen.id
    7879487

· super_gen.engine_type
    GASOLINE

· super_gen.gear_type
    FORWARD_CONTROL

· super_gen.transmission
    MECHANICAL

· super_gen.human_name
    2.3 MT (133 л.с.)

· super_gen.nameplate
  

In [41]:
df_auto_ru_2020.head(2)

,bodyType,brand,color,description,engineDisplacement,enginePower,fuelType,image,mileage,modelDate,model_name,name,numberOfDoors,start_date,productionDate,sell_id,vehicleConfiguration,vehicleTransmission,vendor,Владельцы,ПТС,Привод,Руль,price,price_EUR,price_USD,model_info.code,model_info.name,model_info.ru_name,model_info.nameplate.code,model_info.nameplate.name,model_info.nameplate.semantic_url,super_gen.id,super_gen.displacement,super_gen.engine_type,super_gen.gear_type,super_gen.transmission,super_gen.power,super_gen.power_kvt,super_gen.human_name,super_gen.acceleration,super_gen.clearance_min,super_gen.fuel_rate,super_gen.nameplate,super_gen.clearance_max,super_gen.name,super_gen.electric_range,super_gen.charge_time,super_gen.battery_capacity,complectation_dict.id,complectation_dict.name,complectation_dict.available_options,complectation_dict.vendor_colors,Владение.year,Владение.month,equip_lock,equip_abs,equip_electro-mirrors,equip_airbag-driver,equip_electro-window-front,equip_airbag-passenger,equip_computer,equip_front-seats-heat,equip_immo,equip_electro-window-back,equip_mirrors-heat,equip_usb,equip_wheel-configuration1,equip_aux,equip_airbag-side,equip_esp,equip_ptf,equip_multi-wheel,equip_wheel-leather,equip_12v-socket,equip_wheel-power,equip_front-centre-armrest,equip_bluetooth,equip_isofix,equip_seat-transformation,equip_light-sensor,equip_rain-sensor,equip_wheel-configuration2,equip_cruise-control,equip_airbag-curtain
0,Седан,AUDI,97948F,"Машина на полном ходу\nСостояние хорошее\nС документами полный порядок(кристально чистая)\nВсего было 3 владельца \nПо всем вопросам ПИСАТЬ в личные сообщения (не звонить, я вам скину другой номер...",2.3,133.0,бензин,5,359000,1990.0,100,2.3 MT (133 л.с.),4.0,2020-11-13T23:31:09Z,1993,1101578354,SEDAN MECHANICAL 2.3,MECHANICAL,EUROPEAN,3.0,ORIGINAL,передний,LEFT,106000.0,1161.0,1371.0,100,100,100,,,,7879487,2309.0,GASOLINE,FORWARD_CONTROL,MECHANICAL,133.0,98.0,2.3 MT (133 л.с.),10.2,130.0,8.9,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,Седан,AUDI,CACECB,"Продажа от официального дилера KIA - Компания ""АГАЛАТ"".\n\nГарантированная скидка при покупке в КРЕДИТ 20.000 руб.\nВыгода при ОБМЕНЕ до 50.000 руб.\nАВТОМОБИЛЬ ПРОШЕЛ ПРЕДПРОДАЖНУЮ ПОДГОТОВКУ\nГА...",1.8,90.0,бензин,7,204700,1982.0,100,1.8 MT (90 л.с.),4.0,2020-11-12T09:13:49Z,1984,1101559676,SEDAN MECHANICAL 1.8,MECHANICAL,EUROPEAN,3.0,ORIGINAL,передний,LEFT,44000.0,482.0,569.0,100,100,100,,,,20388757,1781.0,GASOLINE,FORWARD_CONTROL,MECHANICAL,90.0,66.0,1.8 MT (90 л.с.),12.0,133.0,7.2,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


## 6. DeviceStatus 15K

In [42]:
df_devices = prepare.prepare_devices()
summarize(df_devices, "devices")

=== devices ===
shape: (15000, 17)
memory: 9.5 MB
                              dtype  n_unique  n_null  null_%
device_model                 object      9611       0     0.0
model_family                 object        18       0     0.0
manufacturer                 object        10       0     0.0
device_type                  object        19       0     0.0
release_year                  int64        19       0     0.0
status                       object         2       0     0.0
usage_intensity              object         3       0     0.0
climate_zone                 object         3       0     0.0
technical_specs.resolution   object         3   12682    84.5
start_year                  float64        19    7443    49.6
service_life_years          float64         6    7443    49.6
predicted_break_year        float64        24    7443    49.6
actual_break_year           float64        26    7443    49.6
technical_specs.ports       float64         4   11864    79.1
technical_specs.conn

In [43]:
show_samples(df_devices, "devices")

=== samples: devices ===

· device_model
    ПРО-352

· model_family
    ПРО-Series

· manufacturer
    Lenovo

· device_type
    Projector

· status
    in stock

· usage_intensity
    low

· climate_zone
    moderate

· technical_specs.resolution
    Full HD

· technical_specs.connection
    Bluetooth

· technical_specs.type
    Condenser


In [44]:
df_devices.head(2)

,device_model,model_family,manufacturer,device_type,release_year,status,usage_intensity,climate_zone,technical_specs.resolution,start_year,service_life_years,predicted_break_year,actual_break_year,technical_specs.ports,technical_specs.connection,technical_specs.type,technical_specs.dpi
0,ПРО-352,ПРО-Series,Lenovo,Projector,2013,in stock,low,moderate,Full HD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ВЕБ-192,ВЕБ-Series,Logitech,Webcam,2016,in stock,medium,hot,1080p,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Итог

Все 6 датасетов лежат в `data/raw_ru/<name>/clean.parquet`.
Дальше — блокинг кандидатов + разметка (exp 13-стайл LLM-агентом).